In [1]:
import sys
import os
import json
from PyQt6.QtWidgets import QApplication, QTreeView, QVBoxLayout, QWidget, QPushButton, QFileDialog
from PyQt6.QtGui import QStandardItemModel, QStandardItem


# Basisverzeichnis bestimmen (eine Ebene nach oben)
base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
todo_path = os.path.join(base_dir, "data", "todo.json")

# JSON-Datei einlesen
def load_todo():
    if os.path.exists(todo_path):
        with open(todo_path, "r", encoding="utf-8") as file:
            return json.load(file)
    return {"tasks": []}  # Falls keine Datei existiert, leere Struktur zurückgeben

# JSON-Datei speichern
def save_todo(data):
    os.makedirs(os.path.dirname(todo_path), exist_ok=True)
    with open(todo_path, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=4)


# JSON Editor als PyQt6 Widget
class JsonEditor(QWidget):
    def __init__(self, json_data):
        super().__init__()

        self.setWindowTitle("JSON Editor")
        self.setGeometry(100, 100, 600, 400)

        # Hauptlayout
        layout = QVBoxLayout()

        # TreeView erstellen
        self.tree_view = QTreeView()
        layout.addWidget(self.tree_view)

        # Modell für TreeView
        self.model = QStandardItemModel()
        self.tree_view.setModel(self.model)

        # JSON-Daten in die GUI laden
        self.load_json(json_data)

        # Speichern-Button
        save_button = QPushButton("Speichern")
        save_button.clicked.connect(self.save_json)
        layout.addWidget(save_button)

        # Layout setzen
        self.setLayout(layout)

    def load_json(self, data, parent=None):
        if parent is None:
            self.model.clear()
            self.model.setHorizontalHeaderLabels(["Key", "Value"])
            parent = self.model

        if isinstance(data, dict):
            for key, value in data.items():
                item = QStandardItem(key)
                parent.appendRow([item, QStandardItem(str(value))])
                self.load_json(value, item)
        elif isinstance(data, list):
            for index, value in enumerate(data):
                item = QStandardItem(f"[{index}]")
                parent.appendRow([item, QStandardItem(str(value))])
                self.load_json(value, item)
    
    def save_json(self):
        data = self.get_json_from_model(self.model)
        save_todo(data)

    def get_json_from_model(self, model):
        def traverse(item):
            if item.rowCount() == 0:
                return item.text()
            elif item.child(0) and item.child(0).text().startswith("["):
                return [traverse(item.child(row, 1)) for row in range(item.rowCount())]
            else:
                return {item.child(row, 0).text(): traverse(item.child(row, 1)) for row in range(item.rowCount())}

        return traverse(model.invisibleRootItem())


# Anwendung starten
if __name__ == "__main__":
    app = QApplication(sys.argv)
    data = load_todo()
    editor = JsonEditor(data)
    editor.show()
    sys.exit(app.exec())



SystemExit: 0

/Users/Coby/face_recognition/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3558: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
